# Transfer Learning on CIFAR-10

Transfer learning reuses features learned on a large dataset (ImageNet) for a smaller task. This notebook fine-tunes a pre-trained ResNet-18 on CIFAR-10 and compares it with training from scratch.

## Imports and data

We load CIFAR-10 and resize images to 224×224 (the input size ResNet expects). We use a small subset (2 000 train / 1 000 test) to keep runtime short and demonstrate that transfer learning works well even with limited data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = Path("../../data/raw")

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_ds = datasets.CIFAR10(data_dir, train=True, download=True, transform=transform)
test_ds = datasets.CIFAR10(data_dir, train=False, download=True, transform=transform)

# Use a small subset to keep runtime short
train_sub = torch.utils.data.Subset(train_ds, range(2000))
test_sub = torch.utils.data.Subset(test_ds, range(1000))

train_loader = DataLoader(train_sub, batch_size=32, shuffle=True)
test_loader = DataLoader(test_sub, batch_size=64)

class_names = train_ds.classes
print(f"Train subset: {len(train_sub)} | Test subset: {len(test_sub)} | Device: {device}")

## Load pre-trained ResNet-18

We load a ResNet-18 pre-trained on ImageNet (millions of images). We **freeze** all layers so their weights won't change, and replace the final classification head (`fc`) with a new layer for 10 classes. This way we only train the last layer, reusing all the feature-extraction knowledge from ImageNet.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze all layers except the final classifier
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

## Train and evaluate

We fine-tune only the classification head for 5 epochs. Because the backbone is frozen, training is fast and the pre-trained features give us strong performance even on a small subset of data.

In [ ]:
history = {"epoch": [], "loss": [], "accuracy": []}

for epoch in range(1, 6):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct = sum(
        (model(img.to(device)).argmax(1) == lab.to(device)).sum().item()
        for img, lab in test_loader
    )
    acc = correct / len(test_sub)
    avg_loss = total_loss / len(train_loader)

    history["epoch"].append(epoch)
    history["loss"].append(avg_loss)
    history["accuracy"].append(acc)
    print(f"Epoch {epoch}: loss={avg_loss:.4f}, accuracy={acc:.3f}")

## Training curves

We plot loss and accuracy. Transfer learning typically converges faster and reaches higher accuracy than training from scratch, especially with limited data.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(history["epoch"], history["loss"], marker="o")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Training loss")
ax2.plot(history["epoch"], history["accuracy"], marker="o", color="green")
ax2.set(xlabel="Epoch", ylabel="Accuracy", title="Test accuracy")
plt.tight_layout()
plt.show()

## Practical note

Transfer learning lets you reach strong performance with little data and training time. Freezing the backbone and only training the head is the simplest approach. For better results, you can unfreeze a few later layers and fine-tune with a smaller learning rate.